In [1]:
import json

import requests 
from markdownify import markdownify as md
from bs4 import BeautifulSoup
from urllib.parse import quote, urlsplit

In [2]:
API_URL = "https://control.fandom.com/api.php"
BASE_URL = "https://control.fandom.com/wiki/"

ROOT_CATEGORIES = [
    
    "Category:Collectibles_/_Correspondence"
    , "Category:Collectibles_/_Multimedia"
    , "Category:Collectibles_/_Research_&_Records"
    , "Category:Collectibles_/_Case_Files"
    , "Category:Collectibles_/_Hotline"
]

session = requests.Session()
session.headers.update({
    "User-Agent": "ControlCollectiblesResearch/1.0"
})

In [39]:
def get_page_html(title):
    params = {
        "action" : "parse"
        , "format" : "json"
        , "page" : title 
        , "prop" : "text"
        , "redirects": 1
    }
    response = session.get(API_URL, params= params)
    response.raise_for_status()
    data = response.json()
    return data["parse"]["text"]["*"]

In [55]:
title = "Brian's Movie Den Ep. 3"
html = get_page_html(title)
soup = BeautifulSoup(html, "html.parser")

In [56]:
def extract_payload(soup):
    possible_types = {
        "quote": "table.cquote td[style*='font-style:italic']",
        "video_description": "h2 span#Video_description",
        "transcript": [
            "h2 span#Transcript",
            "h2 span#Trancript",
            "h2 span#File_text",
            "h3 span#Transcript",
        ],
        "lyrics": "h2 span#Lyrics",
    }

    for exc, selector in possible_types.items():

        if isinstance(selector, list):
            for i in selector:
                anchor = soup.select_one(i)
                if anchor is None:
                    continue
                else:
                    get_selector_name = i
                    break
        elif isinstance(selector, str):
            get_selector_name = selector
            anchor = soup.select_one(selector)

        if anchor is None:
            continue

        if exc == "quote":
            text = anchor.decode_contents().strip()
            return {"type": exc, "text": text}

        elif exc == "video_description":
            h2 = anchor.find_parent(f"{get_selector_name[:2]}")
            paragraphs = []

            for sibling in h2.find_next_siblings():
                if sibling.name in ["h2", "h3"]:
                    break

                if sibling.name == "p":
                    paragraphs.append(str(sibling).replace("\n", ""))

            if not paragraphs:
                return {}

            return {"type": exc, "text": "<br>".join(paragraphs)}

        elif exc in ["transcript", "lyrics"]:
            h2 = anchor.find_parent(f"{get_selector_name[:2]}")

            transcript_wrapper = h2.find_next("div", class_="va-transcript")

            text = None

            if transcript_wrapper:
                transcript_text = transcript_wrapper.select_one(
                    "div.va-transcript-text"
                )

                if transcript_text:
                    text = transcript_text.decode_contents().replace("\n", "").strip()
            else:
                paragraphs = []
                for sibling in h2.find_next_siblings():
                    if sibling.name in ["h2", "h3"]:
                        break
                    if sibling.name == "p":
                        paragraphs.append(str(sibling).replace("\n", ""))
                text = "<br>".join(paragraphs)
            if text is None:
                return {}
            return {"type": exc, "text": text}

In [59]:
from urllib.parse import unquote

def get_yt_link(file_title: str):
    params = {
        "action": "query",
        "format": "json",
        "prop": "imageinfo",
        "titles": file_title,
        "iiprop": "url|metadata|extmetadata",
    }
    response = session.get(API_URL, params=params)
    response.raise_for_status()
    data = response.json()
    page = next(iter(data["query"]["pages"].values()))
    imageinfo = page["imageinfo"][0]
    metadata = imageinfo.get("metadata", [])

    metadata_dict = {
        item["name"]: item["value"]
        for item in metadata
    }

    if metadata_dict.get("provider") == "youtube":
        video_id = metadata_dict.get("videoId")

        youtube_url = (
            f"https://www.youtube.com/watch?v={video_id}"
            if video_id
            else None
        )
    else:
        youtube_url = None
    
    return youtube_url

def process_blob_link(soup):
    possible_blobs = {
        "audio": "audio[src]"
        , "video": "figure.pi-item a.video-thumbnail"
        , "image":  "figure.pi-item a"
    }

    for exc, selector in possible_blobs.items():
        anchor = soup.select_one(selector)
        if anchor is None:
            continue

        if exc == "audio":
            blob_link = urlsplit(anchor.get("src"))._replace(query= "").geturl()
            return {
                "type": exc
                , "link": blob_link
            }
        elif exc == "video":
            file_url = anchor.get("href")
            img = anchor.select_one("img")
            video_key = img.get("data-video-key") if img else None
            file_title = "File:" + unquote(video_key)

            yt_link = get_yt_link(file_title)
            return {
                "type": exc
                , "link": yt_link
            }
            
        elif exc == "image":
            image_link = urlsplit(anchor.get("href"))._replace(query= "").geturl()
            return {
                "type": exc
                , "link": image_link
            }
        

In [58]:
process_blob_link(soup)

Inside video


{'type': 'video', 'link': 'https://www.youtube.com/watch?v=HAdi5rnC7hk'}

In [31]:
anchor = soup.select_one("figure.pi-item a.video-thumbnail")


<p>Dear <span class=\"new\" data-uncrawlable-url=\"L3dpa2kvTmV3X1lvcmtfVHJpYnVuZT9hY3Rpb249ZWRpdCZyZWRsaW5rPTE=\" title=\"New York Tribune (page does not exist)\">New York Tribune</span>,<br/><br/>Airplanes aren't real. I figured out how they do it.<br/><br/>The windows are TV screens. The whole thing moves on big tracks like a rollercoaster that moves through underground tunnels in the Earth. Airports are more like train stations.<br/><br/>They do this because the sky is full of monsters that they don't want us to know about. The planes we see in the sky are the monsters. The government made the Earth-trains look like the monsters so they could lie to us better.<br/><br/>Don't contact me.</p>

In [26]:
video = soup.select_one(
    "figure.pi-item a.video-thumbnail"
)

if video:
    file_url = video.get("href")

    img = video.select_one("img")
    video_key = img.get("data-video-key") if img else None

    print(file_url)
    print(video_key)

https://control.fandom.com/wiki/File:Control_Sankarin_Tango_(Hero%27s_Tango)
Control_Sankarin_Tango_%28Hero%27s_Tango%29


In [29]:
from urllib.parse import unquote


unquote(video_key)

"Control_Sankarin_Tango_(Hero's_Tango)"

In [31]:
file_title = "File:" + unquote(video_key)

params = {
    "action": "query",
    "format": "json",
    "prop": "imageinfo",
    "titles": file_title,
    "iiprop": "url|metadata|extmetadata",
}

response = session.get(API_URL, params=params)
response.raise_for_status()

data = response.json()

print(
    json.dumps(
        data,
        indent=4
    )
)

{
    "batchcomplete": "",
    "query": {
        "normalized": [
            {
                "from": "File:Control_Sankarin_Tango_(Hero's_Tango)",
                "to": "File:Control Sankarin Tango (Hero's Tango)"
            }
        ],
        "pages": {
            "2141": {
                "pageid": 2141,
                "ns": 6,
                "title": "File:Control Sankarin Tango (Hero's Tango)",
                "imagerepository": "local",
                "imageinfo": [
                    {
                        "url": "https://static.wikia.nocookie.net/control6745/images/9/92/Control_Sankarin_Tango_%28Hero%27s_Tango%29/revision/latest?cb=20200402232107",
                        "descriptionurl": "https://control.fandom.com/wiki/File:Control_Sankarin_Tango_(Hero%27s_Tango)",
                        "descriptionshorturl": "https://control.fandom.com/index.php?curid=2141",
                        "metadata": [
                            {
                                "n

In [8]:
params = {
    "action": "query",
    "format": "json",
    "prop": "imageinfo",
    "titles": file_title,
    "iiprop": "url|extmetadata",
}

response = session.get(API_URL, params=params)
response.raise_for_status()

data = response.json()

print(json.dumps(data, indent=4))

{
    "batchcomplete": "",
    "query": {
        "normalized": [
            {
                "from": "File:Control_Brian's_Movie_Den_Ep._3",
                "to": "File:Control Brian's Movie Den Ep. 3"
            }
        ],
        "pages": {
            "2475": {
                "pageid": 2475,
                "ns": 6,
                "title": "File:Control Brian's Movie Den Ep. 3",
                "imagerepository": "local",
                "imageinfo": [
                    {
                        "url": "https://static.wikia.nocookie.net/control6745/images/5/5f/Control_Brian%27s_Movie_Den_Ep._3/revision/latest?cb=20200525135143",
                        "descriptionurl": "https://control.fandom.com/wiki/File:Control_Brian%27s_Movie_Den_Ep._3",
                        "descriptionshorturl": "https://control.fandom.com/index.php?curid=2475",
                        "extmetadata": {
                            "DateTime": {
                                "value": "2020-05-25T

In [21]:
params = {
    "action": "query",
    "format": "json",
    "prop": "imageinfo",
    "titles": file_title,
    "iiprop": "url|metadata|extmetadata",
}

response = session.get(API_URL, params=params)
response.raise_for_status()

data = response.json()


In [22]:
page = next(iter(data["query"]["pages"].values()))
imageinfo = page["imageinfo"][0]

In [ ]:
page = next(iter(data["query"]["pages"].values()))
imageinfo = page["imageinfo"][0]
metadata = imageinfo.get("metadata", [])

metadata_dict = {
    item["name"]: item["value"]
    for item in metadata
}

if metadata_dict.get("provider") == "youtube":
    video_id = metadata_dict.get("videoId")

    youtube_url = (
        f"https://www.youtube.com/watch?v={video_id}"
        if video_id
        else None
    )
else:
    youtube_url = None

In [24]:
youtube_url

'https://www.youtube.com/watch?v=HAdi5rnC7hk'